# 🚀 Mission Control AI — ARES-1

**Global Solution 2026.1 — Prompt and Artificial Intelligence (FIAP)**

Sistema inteligente de monitoramento de uma missão espacial experimental.
O sistema gera **dados simulados** (temperatura, bateria, geração solar e
comunicação), aplica **lógica de alertas e decisão automática**, e usa o
modelo de linguagem **Llama 3.2 (via Ollama)** para fazer a **análise
operacional** de cada ciclo com base no contexto da missão.

> **Como executar:** rode as células **em ordem, de cima para baixo**.
> A primeira parte instala o Ollama e baixa o modelo (leva ~1–2 min na
> primeira vez). Depois é só rodar a simulação.

## 1. Instalação do Ollama e do modelo Llama

In [ ]:
# 1) Instalar o Ollama no Colab
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# 2) Iniciar o servidor do Ollama em background
import subprocess, time
subprocess.Popen(["ollama", "serve"])
time.sleep(5)   # dá tempo do servidor subir
print("Servidor Ollama iniciado.")

In [ ]:
# 3) Baixar o modelo Llama 3.2 1B (leve e rápido)
!ollama pull llama3.2:1b

In [ ]:
# 4) Instalar a biblioteca Python do Ollama
!pip install ollama -q
print("Biblioteca ollama instalada.")

## 2. Configuração da missão e limites operacionais

Aqui definimos os parâmetros da missão e os **limites (thresholds)** que
separam o que é `NORMAL`, `ATENÇÃO` e `CRÍTICO`.

In [ ]:
import random
from datetime import datetime, timedelta

# Identidade da missão
MISSAO = {
    "nome": "ARES-1",
    "descricao": "Estação orbital experimental em órbita de Marte",
    "modulos": ["Comando", "Energia", "Habitat", "Comunicação"],
}

# Limites operacionais usados pelos alertas
LIMITES = {
    "temperatura":   {"atencao_max": 60, "critico_max": 85,
                      "atencao_min": -20, "critico_min": -40},   # graus C
    "bateria":       {"atencao": 50, "critico": 20},             # %
    "geracao_solar": {"min_aceitavel": 200},                     # Watts
}

print(f"Missão configurada: {MISSAO['nome']} — {MISSAO['descricao']}")

## 3. Classificação dos parâmetros

Cada parâmetro monitorado é classificado em `NORMAL`, `ATENÇÃO` ou `CRÍTICO`.

In [ ]:
def classificar_temperatura(t):
    L = LIMITES["temperatura"]
    if t >= L["critico_max"] or t <= L["critico_min"]:
        return "CRÍTICO"
    if t > L["atencao_max"] or t < L["atencao_min"]:
        return "ATENÇÃO"
    return "NORMAL"

def classificar_bateria(b):
    L = LIMITES["bateria"]
    if b < L["critico"]:
        return "CRÍTICO"
    if b < L["atencao"]:
        return "ATENÇÃO"
    return "NORMAL"

def classificar_comunicacao(status):
    if status == "perdido":
        return "CRÍTICO"
    if status == "instável":
        return "ATENÇÃO"
    return "NORMAL"

## 4. Motor de alertas

Varre a telemetria e gera a lista de alertas ativos no ciclo.

In [ ]:
def gerar_alertas(tel):
    alertas = []
    c_temp = classificar_temperatura(tel["temperatura"])
    c_bat  = classificar_bateria(tel["bateria"])
    c_com  = classificar_comunicacao(tel["comunicacao"])

    if c_temp != "NORMAL":
        alertas.append((c_temp, f"Temperatura em {tel['temperatura']}°C"))
    if c_bat != "NORMAL":
        alertas.append((c_bat, f"Bateria em {tel['bateria']}%"))
    if c_com != "NORMAL":
        alertas.append((c_com, f"Sinal de comunicação {tel['comunicacao']}"))
    if tel["geracao_solar"] < LIMITES["geracao_solar"]["min_aceitavel"]:
        alertas.append(("ATENÇÃO", f"Geração solar baixa: {tel['geracao_solar']}W"))
    return alertas

## 5. Lógica de tomada de decisão automática

Regras determinísticas que decidem ações corretivas. Ex.: **bateria < 20% →
ativar modo de economia**.

In [ ]:
def tomar_decisoes(tel):
    acoes = []
    if tel["bateria"] < LIMITES["bateria"]["critico"]:
        acoes.append("ATIVAR MODO DE ECONOMIA: desligar sistemas não essenciais.")
    if tel["temperatura"] >= LIMITES["temperatura"]["critico_max"]:
        acoes.append("ATIVAR RESFRIAMENTO DE EMERGÊNCIA no módulo afetado.")
    elif tel["temperatura"] <= LIMITES["temperatura"]["critico_min"]:
        acoes.append("ATIVAR AQUECIMENTO DE EMERGÊNCIA no módulo afetado.")
    if tel["comunicacao"] == "perdido":
        acoes.append("ALTERNAR PARA ANTENA DE BACKUP e reorientar para a Terra.")
    if tel["geracao_solar"] < LIMITES["geracao_solar"]["min_aceitavel"] and tel["bateria"] < 50:
        acoes.append("REORIENTAR PAINÉIS SOLARES para maximizar a captação.")
    if not acoes:
        acoes.append("Operação nominal. Nenhuma ação corretiva necessária.")
    return acoes

## 6. Análise com IA (Llama via Ollama)

Aqui está o coração do projeto: a telemetria e os alertas são enviados para
o modelo **Llama 3.2** com um **system prompt** que dá a ele o papel de
*oficial de controle da missão*. A IA devolve uma análise técnica em
linguagem natural.

Há um **fallback** baseado em regras: se o modelo não responder (ex.: timeout),
o sistema ainda produz uma análise — assim a demonstração nunca quebra.

In [ ]:
import ollama

SYSTEM_PROMPT = (
    "Você é o oficial de controle da missão espacial ARES-1, uma estação "
    "orbital experimental em órbita de Marte. Você recebe a telemetria de "
    "cada ciclo (temperatura, bateria, geração solar e comunicação) junto "
    "com os alertas detectados. Sua função é dar uma análise OPERACIONAL "
    "curta e técnica em português: (1) avaliar o estado geral da missão, "
    "(2) apontar o risco mais grave, (3) recomendar a próxima ação. "
    "Responda em no máximo 4 frases, de forma objetiva e profissional."
)

def _analise_fallback(tel, alertas):
    if not alertas:
        return ("Estado geral NOMINAL. Todos os parâmetros dentro da faixa "
                "segura. Recomenda-se manter o monitoramento de rotina.")
    nivel = "CRÍTICO" if any(a[0] == "CRÍTICO" for a in alertas) else "ATENÇÃO"
    return (f"Estado geral {nivel}: {len(alertas)} parâmetro(s) fora do normal. "
            "Aplicar imediatamente as decisões automáticas listadas e priorizar "
            "o alerta de maior severidade.")

def analisar_com_ia(tel, alertas):
    lista_alertas = "; ".join(f"[{n}] {m}" for n, m in alertas) or "nenhum"
    user_msg = (
        f"Telemetria do ciclo:\n"
        f"- Temperatura: {tel['temperatura']}°C\n"
        f"- Bateria: {tel['bateria']}%\n"
        f"- Geração solar: {tel['geracao_solar']}W\n"
        f"- Comunicação: {tel['comunicacao']}\n"
        f"Alertas detectados: {lista_alertas}\n"
        f"Faça a análise operacional."
    )
    try:
        resp = ollama.chat(
            model="llama3.2:1b",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": user_msg},
            ],
        )
        return resp["message"]["content"].strip()
    except Exception as e:
        print(f"[IA indisponível, usando análise de contingência: {e}]")
        return _analise_fallback(tel, alertas)

## 7. Exibição do painel de status

In [ ]:
def exibir_status(ciclo, tel, alertas, acoes, analise_ia):
    print("=" * 64)
    print(f"  MISSION CONTROL AI — {MISSAO['nome']}   |   CICLO {ciclo}")
    print("=" * 64)
    print(f"  Timestamp     : {tel['timestamp']}")
    print(f"  Temperatura   : {tel['temperatura']:>5}°C   [{classificar_temperatura(tel['temperatura'])}]")
    print(f"  Bateria       : {tel['bateria']:>5}%    [{classificar_bateria(tel['bateria'])}]")
    print(f"  Geração solar : {tel['geracao_solar']:>5}W")
    print(f"  Comunicação   : {tel['comunicacao']}   [{classificar_comunicacao(tel['comunicacao'])}]")
    print("-" * 64)
    if alertas:
        print("  ⚠️  ALERTAS:")
        for nivel, msg in alertas:
            print(f"      [{nivel}] {msg}")
    else:
        print("  ✅ ALERTAS: nenhum")
    print("-" * 64)
    print("  🛠️  DECISÕES AUTOMÁTICAS:")
    for a in acoes:
        print(f"      → {a}")
    print("-" * 64)
    print("  🤖 ANÁLISE DA IA (Llama):")
    for linha in analise_ia.split("\n"):
        print(f"      {linha}")
    print("=" * 64)
    print()

## 8. Geração dos dados simulados

Para a demonstração usamos uma sequência de cenários que inclui operação
normal, atenção e **um cenário crítico completo** (ciclo 4). Você também pode
gerar telemetria 100% aleatória com `gerar_telemetria_aleatoria()`.

In [ ]:
def gerar_cenarios():
    base = datetime(2026, 6, 1, 14, 0, 0)
    cenarios = [
        {"temperatura": 24, "bateria": 88, "geracao_solar": 950, "comunicacao": "estável"},
        {"temperatura": 31, "bateria": 72, "geracao_solar": 880, "comunicacao": "estável"},
        {"temperatura": 47, "bateria": 45, "geracao_solar": 410, "comunicacao": "instável"},
        {"temperatura": 92, "bateria": 16, "geracao_solar": 120, "comunicacao": "perdido"},   # CRÍTICO
        {"temperatura": 38, "bateria": 33, "geracao_solar": 600, "comunicacao": "instável"},
        {"temperatura": 22, "bateria": 64, "geracao_solar": 910, "comunicacao": "estável"},
    ]
    for i, c in enumerate(cenarios):
        c["timestamp"] = (base + timedelta(minutes=5 * i)).strftime("%Y-%m-%d %H:%M:%S")
    return cenarios

def gerar_telemetria_aleatoria():
    return {
        "temperatura": random.randint(-50, 100),
        "bateria": random.randint(5, 100),
        "geracao_solar": random.randint(0, 1000),
        "comunicacao": random.choice(["estável", "instável", "perdido"]),
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    }

## 9. Execução do monitoramento

Roda a missão ciclo a ciclo: gera alertas, toma decisões e chama a IA para a
análise de cada ciclo.

In [ ]:
print(f"\n🚀 INICIANDO MONITORAMENTO — Missão {MISSAO['nome']}")
print(f"{MISSAO['descricao']}\n")

for i, tel in enumerate(gerar_cenarios(), start=1):
    alertas  = gerar_alertas(tel)
    acoes    = tomar_decisoes(tel)
    analise  = analisar_com_ia(tel, alertas)
    exibir_status(i, tel, alertas, acoes, analise)

## 10. (Opcional) Modo conversacional

Pergunte qualquer coisa à IA de controle sobre o estado da missão.

In [ ]:
def perguntar_ao_controle(pergunta, tel):
    contexto = (f"Estado atual — Temp: {tel['temperatura']}°C, "
                f"Bateria: {tel['bateria']}%, Solar: {tel['geracao_solar']}W, "
                f"Comunicação: {tel['comunicacao']}.")
    try:
        resp = ollama.chat(model="llama3.2:1b", messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"{contexto}\nPergunta da equipe: {pergunta}"},
        ])
        return resp["message"]["content"].strip()
    except Exception as e:
        return f"[IA indisponível: {e}]"

# Exemplo:
estado = gerar_cenarios()[3]   # o ciclo crítico
print(perguntar_ao_controle("Qual a prioridade número 1 agora?", estado))